# Notebook 3: Teleconnection Analysis

This notebook investigates whether large scale climate modes can explain and predict compound wind hydro energy droughts in Patagonia:

1. Lag correlation analysis between climate modes (SAM, ONI, IOD) and WHDI
2. Seasonal stratification of teleconnections
3. Spatial composite mapping showing how climate modes affect the study region
4. SAM trend analysis and non stationarity implications

**Key finding from Notebook 2:** Wind and runoff anomalies are significantly  correlated in DJF (r=0.27) and MAM (r=0.35) but not in JJA/SON. Compound droughts occur 1.81× more often than random chance. This suggests a shared climate driver active in summer/autumn and this notebook identifies that driver.

## Imports and loading

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats
import os

df = pd.read_csv(
    "../data/processed/whdi_timeseries.csv", index_col="date", parse_dates=True
)

spatial_path = "../data/processed/era5_masked_spatial.nc"
file_size_mb = os.path.getsize(spatial_path) / (1024 * 1024)
print(f"Spatial file size: {file_size_mb:.1f} MB")

# Load spatial ERA5 for the maps
ds = xr.open_dataset("../data/processed/era5_masked_spatial.nc")

print(f"Timeseries data: {df.shape}")
print(f"Period: {df.index[0]}-{df.index[-1]}")
print(f"Columns: {list(df.columns)}")
print(f"Spatial data variables: {list(ds.data_vars)}")

Spatial file size: 27.6 MB
Timeseries data: (564, 13)
Period: 1979-01-01 00:00:00-2025-12-01 00:00:00
Columns: ['wind_speed', 'temperature', 'precipitation', 'runoff', 'snowmelt', 'sam', 'oni', 'iod', 'wind_std', 'runoff_std', 'whdi_1', 'whdi_3', 'whdi_6']
Spatial data variables: ['wind_speed', 'temperature', 'precipitation', 'runoff', 'snowmelt']


## Creating effective degrees of freedom function

With 564 monthly observations, even a tiny correlation (r=0.08) would appear significant at p<0.05 using the raw sample size. But consecutive months are not independent, a positive SAM in January is likely followed by positive SAM in February. The effective degrees of freedom correction accounts for this autocorrelation, giving honest significance tests.

The difference can be dramatic. If both series have lag1 autocorrelation of 0.5, effective n drops from 564 to roughly 188. A correlation that looked significant at n=564 might not be at n=188.

In [ ]:
def effective_n(x, y):
    """
    Compute effective degrees of freedom for correlation significance testing.

    Formula: n_eff = n * (1 - r1_x * r1_y) / (1 + r1_x * r1_y) where r1_x and r1_y are the lag1 autocorrelations of x and y.
    """
    n = len(x)

    # lag1 autocorrelation for each series
    r1_x = pd.Series(x).autocorr(lag=1)
    r1_y = pd.Series(y).autocorr(lag=1)

    # Handle nan autocorrelation
    if np.isnan(r1_x) or np.isnan(r1_y):
        return n

    n_eff = n * (1 - (r1_x * r1_y)) / (1 + (r1_x * r1_y))

    return max(int(n_eff), 3)


def correlation_with_significance(x, y):
    """
    Compute Pearson correlation with significance adjusted for autocorrelation.

    Returns correlation coefficient, raw p-value, adjusted p-value, and effective sample size.
    """
    # Remove nan pairs
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = np.array(x)[mask]
    y_clean = np.array(y)[mask]

    if len(x_clean) < 4:
        return np.nan, np.nan, np.nan, 0

    # Correlation
    r, p_raw = stats.pearsonr(x_clean, y_clean)

    # Adjusted significance using effective degrees of freedom
    n_eff = effective_n(x_clean, y_clean)

    # Recompute p value with effective degrees of freedom
    # t statistic for correlaton
    if abs(r) >= 1.0:
        p_adj = 0.0
    else:
        t_stat = r * np.sqrt((n_eff - 2) / (1 - r**2))
        p_adj = 2 * stats.t.sf(abs(t_stat), df=n_eff - 2)

    return r, p_raw, p_adj, n_eff